# Stage 3 — Aggregate, score, plot

Reads per-`(model, benchmark)` JSONs from `results/`, writes the aggregated long-format Parquets, the bootstrap-CI table, and all figures. Re-runnable in seconds — change `plotting.py` or `statistics.py` and re-execute without touching the GPU stage.

In [ ]:
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive; drive.mount('/content/drive')
    %cd /content/llm-bias-eval
    if not os.path.lexists('results'):
        os.symlink('/content/drive/MyDrive/llm-bias-eval/results', 'results')

In [ ]:
from pathlib import Path
import pandas as pd
import yaml

from biaseval.analysis.aggregate_results import (
    aggregate_logit_results, aggregate_probe_results,
    aggregate_intervention_results, write_aggregated,
    cross_pair_direction_cosines,
)
from biaseval.analysis.plotting import generate_all
from biaseval.analysis.statistics import (
    per_example_bootstrap,
    pair_significance_table,
    pair_significance_per_language,
    cross_benchmark_consistency,
    cross_language_consistency,
)
from biaseval.analysis.regression import write_regression_report

RESULTS = Path('results')
FIGURES = Path('figures')
TABLES = Path('results/tables'); TABLES.mkdir(parents=True, exist_ok=True)

with open('configs/models.yaml') as f:
    _ycfg = yaml.safe_load(f)
registry_pairs = [
    (entry['base_id'], entry['instruct_id'], fam_name, gen['name'], entry['size'])
    for fam_name, family in _ycfg['families'].items()
    for gen in family['generations']
    for entry in gen['models']
]
print(f'{len(registry_pairs)} (base, instruct) pairs in registry')

In [ ]:
write_aggregated(RESULTS, RESULTS / 'aggregated.parquet')
logit_df = aggregate_logit_results(RESULTS)
probe_df = aggregate_probe_results(RESULTS)
intv_df  = aggregate_intervention_results(RESULTS)
print(f'logit={len(logit_df)}  probe={len(probe_df)}  intervention={len(intv_df)}')

In [ ]:
rows = [
    {'benchmark': bench, 'model_id': model_id, 'prompt_mode': prompt_mode, **stats}
    for bench in ('crows_pairs', 'stereoset')
    for (model_id, prompt_mode), stats in per_example_bootstrap(RESULTS, bench, n_iter=1000).items()
]
bootstrap_df = pd.DataFrame(rows)
bootstrap_df.to_csv(TABLES / 'bootstrap_ci.csv', index=False)

## Søgaard-aligned tables: paired stats, cross-benchmark consistency, direction rotation

Built **before** `generate_all` so the headline figures (forest plot, cross-benchmark heatmap, direction-rotation curves) can pick them up. Each one returns an empty DataFrame if the underlying results aren't on disk yet — safe to re-run anytime.

In [ ]:
# Per-pair CrowS-Pairs paired stats (Søgaard 2013/2014: effect size first,
# permutation tests, two thresholds α=0.05 and α=0.0025).
pair_sig_df = pair_significance_table(
    RESULTS, registry_pairs, prompt_mode='raw', scoring='norm',
    n_perm=10_000, n_boot=10_000, seed=42,
)
pair_sig_df.to_csv(TABLES / 'pair_significance.csv', index=False)

# Cross-benchmark agreement: per pair, does the alignment effect have the
# same sign across CrowS / BBQ / StereoSet / IAT?
consistency_df, corr_df = cross_benchmark_consistency(RESULTS, registry_pairs)
consistency_df.to_csv(TABLES / 'cross_benchmark_consistency.csv', index=False)
corr_df.to_csv(TABLES / 'cross_benchmark_spearman.csv')

# Probe-direction rotation: cos(direction_base[L], direction_instruct[L]) per
# pair, attribute, layer. Drops to 0 → alignment rotated the demographic
# direction; stays at 1 → representation untouched.
direction_cosines_df = cross_pair_direction_cosines(RESULTS, registry_pairs)
direction_cosines_df.to_csv(TABLES / 'direction_cosines.csv', index=False)

# Per-language paired stats + cross-language consistency. Each call silently
# skips languages whose JSONs aren't on disk yet, so this is safe to run
# against partial multilingual sweeps.
LANGS = ('en', 'fr', 'es', 'de', 'pt', 'it')
pair_sig_per_lang_df = pair_significance_per_language(
    RESULTS, registry_pairs, languages=LANGS,
    n_perm=10_000, n_boot=10_000, seed=42,
)
pair_sig_per_lang_df.to_csv(TABLES / 'pair_significance_per_language.csv', index=False)

lang_consistency_df, lang_corr_df = cross_language_consistency(
    RESULTS, registry_pairs, languages=LANGS,
)
lang_consistency_df.to_csv(TABLES / 'cross_language_consistency.csv', index=False)
lang_corr_df.to_csv(TABLES / 'cross_language_spearman.csv')

print(f'pair_sig: {len(pair_sig_df)} / {len(registry_pairs)} pairs')
print(f'consistency: {len(consistency_df)} pairs')
print(f'direction cosines: {len(direction_cosines_df)} (pair, attr, layer) rows')
print(f'pair_sig_per_lang: {len(pair_sig_per_lang_df)} (pair × language) rows')
print(f'lang_consistency: {len(lang_consistency_df)} pairs across '
      f'{len([c for c in lang_consistency_df.columns if c.startswith("delta_")])} langs')

In [ ]:
paths = generate_all(
    logit_df, probe_df, figures_dir=FIGURES, intv_df=intv_df,
    pair_sig_df=pair_sig_df,
    consistency_df=consistency_df,
    direction_cosines_df=direction_cosines_df,
    pair_sig_per_lang_df=pair_sig_per_lang_df,
    lang_consistency_df=lang_consistency_df,
    lang_corr_df=lang_corr_df,
    results_dir=RESULTS,
    registry_pairs=registry_pairs,
)
print(f'wrote {len(paths)} figure(s)')

## Confound-controlled regression

Per the supervisor's framing question: estimate the variant (base→instruct) effect on each headline bias metric **after partialling out** model size (log₁₀ params), family, generation, and prompt_mode. Cluster-robust SEs on `model_id` plus a per-pair logistic GEE on CrowS-Pairs. Holm-Bonferroni across the 4 benchmark fits.

Writes `results/tables/regression_report.md` (paste-ready for the thesis) and `regression_summaries.json` (raw fits for downstream slicing).

In [ ]:
md_path = write_regression_report(RESULTS, TABLES)
print(open(md_path).read())

## Inspection: Søgaard-aligned headline tables

Display the three tables built earlier. The pair-significance frame is the headline statistical evidence; cross-benchmark consistency tests whether the alignment effect generalises across our 4 benchmarks; direction cosines summarise the geometric rotation per layer.

In [ ]:
# Pair-significance: sorted by |cohens_d_paired| descending. ★ = reject_holm
# (α=0.05), ★★ = reject_strict (α=0.0025, Søgaard et al. 2014).
pair_sig_df.round(3)

In [ ]:
# Cross-benchmark consistency: Δ in lower-is-less-biased frame, plus
# n_benchmarks_agreeing (out of 4) and `all_agree` flag.
display(consistency_df.round(3))
print('Spearman correlation of Δ across benchmarks (each pair = one obs):')
corr_df.round(3)

In [ ]:
# Quick spot-checks on the new benchmarks (BBQ deferral decomposition,
# implicit/explicit gap, jailbreak reactivation). Each block returns an empty
# slice if the underlying runs aren't on disk yet.

bbq_decomp = (logit_df[(logit_df['benchmark'] == 'bbq')
                       & logit_df['metric'].isin(
                           ['overall_bias_ambig', 'overall_deferral_rate',
                            'overall_conditional_bias'])]
              .pivot_table(index=['family', 'size', 'variant'],
                           columns='metric', values='value'))
print('BBQ deferral decomposition (rows: model × variant):')
display(bbq_decomp.round(3))

ie_gap = (logit_df[logit_df['benchmark'].isin(
              ['implicit_explicit_race', 'implicit_explicit_gender'])
                   & (logit_df['metric'] == 'implicit_explicit_gap')]
          .pivot_table(index=['family', 'size', 'variant'],
                       columns='benchmark', values='value'))
print('\nImplicit−explicit gap by attribute:')
display(ie_gap.round(2))

jb = (logit_df[(logit_df['benchmark'] == 'crows_pairs')
               & (logit_df['metric'] == 'overall')
               & (logit_df['variant'] == 'instruct')
               & logit_df['prompt_mode'].isin(['raw', 'instruct', 'jailbreak'])]
      .pivot_table(index=['family', 'size'],
                   columns='prompt_mode', values='value'))
print('\nCrowS-Pairs overall under raw/instruct/jailbreak (instruct models):')
display(jb.round(2))

## Multilingual stats inspection

Per-language paired stats (pair × language → d, CIs, p_value, Holm) and the cross-language consistency frame. The Spearman matrix below answers "does the alignment effect transfer across languages?"

In [ ]:
# Per-language paired stats. Holm correction is applied within each language
# (so ★ markers compare to the same pair family in that language).
display(pair_sig_per_lang_df.round(3))

# Cross-language consistency: per pair, sign of Δ in each language.
display(lang_consistency_df.round(3))

print('Spearman correlation of paired Δ across languages '
      '(each pair = one observation):')
lang_corr_df.round(3)